# Demo 1

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import rand
from datetime import datetime
import time

spark = SparkSession.builder.appName("MarketDataIngestion").getOrCreate()
try:
    start_time = time.time()

    # Generate 10 million rows of simulated market data
    df = spark.range(10000000).withColumn("time", rand()).withColumn("sym", rand()).withColumn("price", rand()*10+100).withColumn("size", rand()*1000)

    # Write to memory-backed dataframe
    df.count()  # Trigger caching
finally:
    print(f"Ingestion time: {time.time() - start_time:.2f} seconds")
    spark.stop()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/03/24 20:20:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Ingestion time: 3.28 seconds


# Demo 2

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, expr
import time
import pandas as pd
from datetime import datetime, timedelta
import random

# Initialize Spark
spark = SparkSession.builder.appName("VWAP Calculation").getOrCreate()

try:
    # Generate 10M trades
    n = 10000000
    start_time = datetime.now()

    data = pd.DataFrame({
        "time": [start_time + timedelta(seconds=random.randint(0, 3600)) for _ in range(n)],
        "sym": random.choices(["AAPL", "GOOG", "MSFT", "TSLA"], k=n),
        "price": [100 + random.uniform(0, 10) for _ in range(n)],
        "size": [random.randint(10, 1000) for _ in range(n)]
    })

    # Convert to Spark DataFrame
    df = spark.createDataFrame(data)
    df.show(5)

    start_time = time.time()

    # Extract minute from timestamp
    df = df.withColumn("minute", expr("minute(time)"))

    # Compute VWAP
    vwap_df = df.groupBy("minute").agg((sum(col("price") * col("size")) / sum("size")).alias("vwap"))

    # Show result
    vwap_df.show()
finally:
    print("Execution time:", time.time() - start_time, "seconds")
    spark.stop()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/03/24 20:21:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


TypeError: unsupported operand type(s) for -: 'float' and 'datetime.datetime'

25/03/24 21:21:51 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 167264 ms exceeds timeout 120000 ms
25/03/24 21:21:51 WARN SparkContext: Killing executors is not supported by current scheduler.
25/03/24 21:21:54 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$

The Demo 2 doesn't even finish running